# Clase 132 — Localización, detección y segmentación

Las cuatro tareas de visión: **clasificación** (qué), **localización** (1
objeto + caja), **detección** (N objetos + cajas) y **segmentación** (máscara
por píxel). Evolución: Faster R-CNN (two-stage) → YOLO (one-stage) → DETR
(Transformer end-to-end) → YOLOv11 → SAM/SAM 2 (segmentación promptable). Aquí
implementamos a mano las métricas clave (**IoU**, **NMS**, precisión/recall) y
esbozamos el uso de los modelos preentrenados.

Requiere: `numpy`, `tensorflow` / `keras`. Las librerías `ultralytics`,
`transformers` y `segment_anything` se muestran de forma conceptual.

## 1. Formatos de bounding box

Dos convenciones habituales: `(x1, y1, x2, y2)` (esquinas) y `(cx, cy, w, h)`
(centro + tamaño). Convertir entre ambas es rutina.

In [ ]:
import numpy as np
np.random.seed(42)

def xywh_a_xyxy(cx, cy, w, h):
    return np.array([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])

def xyxy_a_xywh(x1, y1, x2, y2):
    return np.array([(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1])

caja = xywh_a_xyxy(50, 50, 20, 30)
print("(cx,cy,w,h)=(50,50,20,30) -> (x1,y1,x2,y2) =", caja)
print("de vuelta a (cx,cy,w,h)   =", xyxy_a_xywh(*caja))

## 2. IoU (Intersection over Union) a mano

`IoU = area(intersección) / area(unión)`. Se considera *match* si `IoU > 0.5`.

In [ ]:
def iou(a, b):
    # cajas en formato (x1, y1, x2, y2)
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

gt = [10, 10, 50, 50]
pred = [20, 20, 60, 60]
print(f"IoU = {iou(gt, pred):.3f}")            # ~0.391
print("match (IoU > 0.5):", iou(gt, pred) > 0.5)

## 3. Non-Maximum Suppression (NMS)

Post-procesado que elimina cajas superpuestas dejando la de mayor score. YOLO
lo aplica internamente; DETR no lo necesita.

In [ ]:
def nms(cajas, scores, umbral_iou=0.5):
    idx = list(np.argsort(scores)[::-1])   # de mayor a menor score
    conservadas = []
    while idx:
        actual = idx.pop(0)
        conservadas.append(actual)
        idx = [i for i in idx if iou(cajas[actual], cajas[i]) < umbral_iou]
    return conservadas

cajas = np.array([[10, 10, 50, 50],
                  [12, 12, 52, 52],
                  [100, 100, 140, 140]], dtype=float)
scores = np.array([0.9, 0.8, 0.7])
keep = nms(cajas, scores)
print("cajas conservadas tras NMS:", keep)   # [0, 2]: la 1 solapa con la 0

## 4. Precisión y recall a IoU=0.5 (base del mAP)

Con las predicciones ordenadas por score, cada una es TP (matchea un GT no
usado con IoU≥0.5) o FP. El AP integra la curva precisión-recall.

In [ ]:
gt_cajas = np.array([[10, 10, 50, 50], [100, 100, 140, 140]], dtype=float)
pred_cajas = np.array([[12, 12, 52, 52],
                       [60, 60, 90, 90],
                       [102, 102, 138, 138]], dtype=float)
pred_scores = np.array([0.9, 0.75, 0.6])

orden = np.argsort(pred_scores)[::-1]
usados = set(); tp = []; fp = []
for i in orden:
    ious = [iou(pred_cajas[i], g) for g in gt_cajas]
    j = int(np.argmax(ious))
    if ious[j] >= 0.5 and j not in usados:
        usados.add(j); tp.append(1); fp.append(0)
    else:
        tp.append(0); fp.append(1)

tp_acc = np.cumsum(tp); fp_acc = np.cumsum(fp)
precision = tp_acc / (tp_acc + fp_acc)
recall = tp_acc / len(gt_cajas)
print("precisión acumulada:", np.round(precision, 2))
print("recall acumulado   :", np.round(recall, 2))

## 5. Pipelines preentrenados (conceptual)

YOLO (one-stage rápido), DETR (Transformer end-to-end) y SAM (segmentación
promptable con puntos/cajas). Se muestran sin ejecutar por dependencias
externas.

In [ ]:
# --- YOLOv11 (Ultralytics) ---
# from ultralytics import YOLO
# modelo = YOLO("yolo11n.pt")
# resultados = modelo("imagen.jpg")        # detección, NMS interno

# --- DETR (Hugging Face Transformers) ---
# from transformers import pipeline
# detector = pipeline("object-detection", model="facebook/detr-resnet-50")

# --- Segment Anything (SAM 2) ---
# from segment_anything import sam_model_registry, SamPredictor
# sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b.pth")

print("YOLO = one-stage rapido | DETR = Transformer end-to-end | "
      "SAM = segmentacion promptable")

## 6. Backbone + heads: anatomía de un detector

Un detector reutiliza un backbone CNN preentrenado y le agrega *heads*: una para
clasificar la caja y otra para regresar sus coordenadas.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

backbone = keras.applications.ResNet50V2(
    weights=None, include_top=False, input_shape=(224, 224, 3))
x = layers.GlobalAveragePooling2D()(backbone.output)
clase = layers.Dense(3, activation="softmax", name="clase")(x)   # N clases
caja = layers.Dense(4, activation="sigmoid", name="caja")(x)     # (x,y,w,h) norm
detector = keras.Model(backbone.input, [clase, caja],
                       name="detector_didactico")
print("salidas del detector:", [o.shape for o in detector.outputs])

## Ejercicios

1. **IoU a mano**: implementá `iou(a, b)` y verificá que para dos cajas
   `40×40` desplazadas 10 px da ≈ 0.39.
2. **NMS**: dado un set con dos cajas muy solapadas y una lejana, comprobá que
   NMS conserva la de mayor score y la lejana.
3. **Precisión-recall**: calculá TP/FP a IoU=0.5 sobre un set pequeño de
   predicciones y GT, y reportá la precisión y recall acumulados.
4. **Detector**: construí un backbone `ResNet50V2(include_top=False)` con dos
   heads (clase y caja) y verificá los shapes de salida.

## Conclusiones

- Las 4 tareas: clasificación, localización, detección y segmentación (pixel-wise).
- **IoU** mide el solape caja-a-caja; es la base de `mAP` y del matching de detección.
- **NMS** elimina cajas redundantes; YOLO lo hace interno, DETR no lo necesita.
- Familia de modelos: Faster R-CNN (two-stage) → YOLO (rápido) → DETR → SAM (promptable).
- Un detector = backbone CNN preentrenado + heads de clasificación y regresión de caja.